# SQL analysis

In [10]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../yelp.db")

## Data checks

In [11]:
pd.read_sql("SELECT COUNT(*) AS total_businesses FROM business;", conn)

,total_businesses
0,150346


In [12]:
pd.read_sql("SELECT COUNT(*) AS missing_categories FROM business WHERE categories IS NULL;", conn)

,missing_categories
0,103


In [13]:
pd.read_sql("""
SELECT COUNT(*) AS missing_stars_or_reviews
FROM business
WHERE stars IS NULL OR review_count IS NULL;
""", conn)

,missing_stars_or_reviews
0,0


In [14]:
pd.read_sql("""
SELECT
    MIN(review_count) AS min_reviews,
    AVG(review_count) AS avg_reviews,
    MAX(review_count) AS max_reviews
FROM business;
""", conn)

,min_reviews,avg_reviews,max_reviews
0,5,44.866561,7568


## Do businesses with more reviews have better ratings?

In [15]:
pd.read_sql("""
SELECT
    -- group businesses by number of reviews
    CASE
        WHEN review_count < 10  THEN '1. 5-9 reviews'
        WHEN review_count < 25  THEN '2. 10-24 reviews'
        WHEN review_count < 100 THEN '3. 25-99 reviews'
        WHEN review_count < 500 THEN '4. 100-499 reviews'
        ELSE '5. 500+ reviews'
    END AS review_bucket,
    COUNT(*) AS nb_businesses,
    ROUND(AVG(stars), 2) AS avg_stars
FROM business
GROUP BY review_bucket
ORDER BY review_bucket;  -- numbers in labels keep the right order
""", conn)

,review_bucket,nb_businesses,avg_stars
0,1. 5-9 reviews,51103,3.61
1,2. 10-24 reviews,46864,3.53
2,3. 25-99 reviews,37732,3.58
3,4. 100-499 reviews,13293,3.79
4,5. 500+ reviews,1354,3.99


Yes: around 3.5-3.6 stars under 100 reviews, almost 4 stars above 500.